In [1]:
!pip install ipywidgets numpy pillow


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
from PIL import Image
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, HTML

# ── Resolution ──────────────────────────────────────────────────────────────
WIDTH, HEIGHT = 320, 240
DISPLAY_WIDTH  = 640
DISPLAY_HEIGHT = 480

# ── Fixed-point parameters ───────────────────────────────────────────────────
MAX_ITER  = 30
SCALE     = 4096
FRAC_BITS = 12

RE_MIN, RE_MAX = -2.0,  2.0
IM_MIN, IM_MAX = -1.5,  1.5

DEFAULT_STEP = int(round((RE_MAX - RE_MIN) * SCALE / (WIDTH - 1)))
DEFAULT_ZR0  = int(round(RE_MIN * SCALE))
DEFAULT_ZI0  = int(round(IM_MIN * SCALE))

ROOTS_F = [
    ( 1.0,        0.0      ),
    (-0.5,        0.8660254),
    (-0.5,       -0.8660254)
]
ROOTS = [(int(round(r * SCALE)), int(round(i * SCALE))) for r, i in ROOTS_F]

# Root colours: red, teal, blue
COL = np.array([
    [230,  57,  70],
    [ 42, 157, 143],
    [ 69, 123, 157]
], dtype=int)

COL_HEX = ['#E63946', '#2A9D8F', '#457B9D']
ROOT_LABELS = ['1 + 0i', '-0.5 + 0.866i', '-0.5 - 0.866i']

TOL       = int(round(0.03 * SCALE))
DENOM_MIN = 1

# ── Fixed-point arithmetic ───────────────────────────────────────────────────
def tdiv(a, b):
    if b == 0:
        return 0
    q = abs(a) // abs(b)
    return -q if (a < 0) != (b < 0) else q

def mul(a, b):
    return tdiv(a * b, SCALE)

# ── View helpers ─────────────────────────────────────────────────────────────
def view_to_registers(zoom_value=1.0, pan_x_value=0, pan_y_value=0):
    step = max(1, int(round(DEFAULT_STEP / zoom_value)))
    centre_r = DEFAULT_ZR0 + (WIDTH  // 2) * DEFAULT_STEP + pan_x_value * step
    centre_i = DEFAULT_ZI0 + (HEIGHT // 2) * DEFAULT_STEP + pan_y_value * step
    zr0 = centre_r - (WIDTH  // 2) * step
    zi0 = centre_i - (HEIGHT // 2) * step
    return int(zr0), int(zi0), int(step)

def registers_to_float_bounds(zr0, zi0, step):
    """Convert Q12 register values to floating-point complex plane bounds."""
    re_left  = zr0  / SCALE
    re_right = (zr0 + (WIDTH  - 1) * step) / SCALE
    im_top   = zi0  / SCALE
    im_bot   = (zi0 + (HEIGHT - 1) * step) / SCALE
    return re_left, re_right, im_top, im_bot

# ── Fractal renderer ─────────────────────────────────────────────────────────
def generate_image(zoom_value=1.0, pan_x_value=0, pan_y_value=0, max_iter=MAX_ITER):
    zr0, zi0, step = view_to_registers(zoom_value, pan_x_value, pan_y_value)
    img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    for py in range(HEIGHT):
        for px in range(WIDTH):
            zr = zr0 + px * step
            zi = zi0 + py * step
            which = -1
            it    = 0

            for it in range(max_iter):
                zr2 = mul(zr, zr) - mul(zi, zi)
                zi2 = tdiv(2 * zr * zi, SCALE)
                zr3 = mul(zr2, zr) - mul(zi2, zi)
                zi3 = mul(zr2, zi) + mul(zi2, zr)
                fr  = zr3 - SCALE
                fi  = zi3
                fpr = 3 * zr2
                fpi = 3 * zi2
                denom = mul(fpr, fpr) + mul(fpi, fpi)
                if denom <= DENOM_MIN:
                    break
                numr = mul(fr, fpr) + mul(fi, fpi)
                numi = mul(fi, fpr) - mul(fr, fpi)
                dr   = tdiv(numr * SCALE, denom)
                di   = tdiv(numi * SCALE, denom)
                zr  -= dr
                zi  -= di
                for k, (rr, ri) in enumerate(ROOTS):
                    if abs(zr - rr) < TOL and abs(zi - ri) < TOL:
                        which = k
                        break
                if which >= 0:
                    break

            if which < 0:
                img[py, px] = (0, 0, 0)
            else:
                shade256 = max(64, 256 - (it * 256) // max_iter)
                img[py, px] = (
                    (int(COL[which][0]) * shade256) >> 8,
                    (int(COL[which][1]) * shade256) >> 8,
                    (int(COL[which][2]) * shade256) >> 8
                )
    return img

def to_png(img_array):
    buf = BytesIO()
    Image.fromarray(np.asarray(img_array, dtype=np.uint8)).save(buf, format='PNG')
    return buf.getvalue()

# ── Info panel HTML ──────────────────────────────────────────────────────────
def make_info_html(zoom_value, pan_x_value, pan_y_value, max_iter, zr0, zi0, step):
    re_l, re_r, im_t, im_b = registers_to_float_bounds(zr0, zi0, step)
    re_width  = re_r - re_l
    im_height = im_b - im_t

    # Closest root to centre of view
    centre_r = (re_l + re_r) / 2
    centre_i = (im_t + im_b) / 2
    dists = [((centre_r - rx)**2 + (centre_i - iy)**2)**0.5
             for rx, iy in ROOTS_F]
    nearest_idx  = int(np.argmin(dists))
    nearest_dist = dists[nearest_idx]

    # Convergence speed note
    if max_iter <= 10:
        iter_note = 'Low — coarse boundaries, fast render'
    elif max_iter <= 25:
        iter_note = 'Medium — balanced detail and speed'
    else:
        iter_note = 'High — sharp boundaries, slower render'

    legend_html = ''.join(
        f'<span style="display:inline-block;width:14px;height:14px;'
        f'background:{COL_HEX[i]};border-radius:3px;margin-right:5px;"></span>'
        f'Root {i+1}: z = {ROOT_LABELS[i]}&nbsp;&nbsp;&nbsp;'
        for i in range(3)
    )

    html = f"""
    <div style="font-family:monospace; font-size:13px; background:#1e1e2e;
                color:#cdd6f4; padding:14px 18px; border-radius:8px;
                margin-top:8px; line-height:1.9;">

      <div style="font-size:15px; font-weight:bold; color:#89b4fa;
                  margin-bottom:8px;">Newton Fractal — f(z) = z&sup3; &minus; 1</div>

      <table style="width:100%; border-collapse:collapse;">
        <tr>
          <td style="color:#a6e3a1; padding-right:20px;">&#x1F50D; Complex plane view</td>
          <td>Re [{re_l:+.4f}, {re_r:+.4f}] &nbsp; Im [{im_t:+.4f}, {im_b:+.4f}]</td>
        </tr>
        <tr>
          <td style="color:#a6e3a1;">&#x1F4CF; View width</td>
          <td>Re span = {re_width:.4f} &nbsp; Im span = {im_height:.4f}</td>
        </tr>
        <tr>
          <td style="color:#a6e3a1;">&#x1F50E; Zoom</td>
          <td>{zoom_value:.2f}x &nbsp;(pixel step = {step/SCALE:.5f} per pixel)</td>
        </tr>
        <tr>
          <td style="color:#a6e3a1;">&#x1F504; Max iterations</td>
          <td>{max_iter} &nbsp;&mdash;&nbsp; {iter_note}</td>
        </tr>
        <tr>
          <td style="color:#a6e3a1;">&#x1F4CC; Nearest root to centre</td>
          <td>Root {nearest_idx+1}: z = {ROOT_LABELS[nearest_idx]} &nbsp;
              (distance = {nearest_dist:.4f})</td>
        </tr>
        <tr>
          <td style="color:#a6e3a1;">&#x2699;&#xFE0F; FPGA registers (Q12)</td>
          <td>ZR0={zr0} &nbsp; ZI0={zi0} &nbsp; STEP={step} &nbsp; MAXIT={max_iter}</td>
        </tr>
      </table>

      <div style="margin-top:10px; padding-top:8px;
                  border-top:1px solid #45475a; font-size:12px;">
        <span style="color:#a6e3a1;">&#x1F3A8; Colour legend &mdash; convergence basin:</span><br/>
        <div style="margin-top:5px;">{legend_html}</div>
        <div style="margin-top:4px; color:#bac2de;">
          Brightness = convergence speed. Brighter = fewer iterations needed.
          Black = did not converge within {max_iter} iterations.
        </div>
      </div>

      <div style="margin-top:10px; padding-top:8px;
                  border-top:1px solid #45475a; font-size:12px; color:#bac2de;">
        <span style="color:#a6e3a1;">&#x1F9EE; Newton's method:</span>
        Each pixel z&#x2080; iterates &nbsp;
        z&#x2099;&#x208A;&#x2081; = z&#x2099; &minus; f(z&#x2099;)/f'(z&#x2099;)
        &nbsp; where &nbsp; f(z) = z&sup3;&minus;1, &nbsp; f'(z) = 3z&sup2;.
        Colour shows which of the 3 roots it converges to.
      </div>

    </div>
    """
    return html

# ── Widgets ──────────────────────────────────────────────────────────────────
image_widget = widgets.Image(value=b'', format='png',
                             width=DISPLAY_WIDTH, height=DISPLAY_HEIGHT)

zoom_slider  = widgets.FloatSlider(value=1.0, min=0.5, max=20.0, step=0.1,
                                   description='Zoom',     continuous_update=False)
pan_x_slider = widgets.IntSlider(  value=0,   min=-320, max=320,  step=1,
                                   description='Pan X',    continuous_update=False)
pan_y_slider = widgets.IntSlider(  value=0,   min=-240, max=240,  step=1,
                                   description='Pan Y',    continuous_update=False)
maxit_slider = widgets.IntSlider(  value=MAX_ITER, min=1, max=63, step=1,
                                   description='Max Iter', continuous_update=False)

apply_button = widgets.Button(description='Apply View',
                              button_style='primary')
reset_button = widgets.Button(description='Reset View')
status       = widgets.Label(value='Press Apply View to render.')
info_output  = widgets.Output()

# ── Render ───────────────────────────────────────────────────────────────────
def render(zoom_value, pan_x_value, pan_y_value, max_iter):
    status.value = 'Rendering...'
    img = generate_image(zoom_value, pan_x_value, pan_y_value, max_iter)
    image_widget.value = to_png(img)
    zr0, zi0, step = view_to_registers(zoom_value, pan_x_value, pan_y_value)
    status.value = 'Ready.'
    with info_output:
        info_output.clear_output(wait=True)
        display(HTML(make_info_html(zoom_value, pan_x_value, pan_y_value,
                                    max_iter, zr0, zi0, step)))

def apply_view(btn):
    render(zoom_slider.value, pan_x_slider.value,
           pan_y_slider.value, maxit_slider.value)

def reset_view(btn):
    zoom_slider.value  = 1.0
    pan_x_slider.value = 0
    pan_y_slider.value = 0
    maxit_slider.value = MAX_ITER
    render(1.0, 0, 0, MAX_ITER)

apply_button.on_click(apply_view)
reset_button.on_click(reset_view)

# ── Layout ───────────────────────────────────────────────────────────────────
display(
    image_widget,
    zoom_slider,
    pan_x_slider,
    pan_y_slider,
    maxit_slider,
    widgets.HBox([apply_button, reset_button]),
    status,
    info_output
)

render(1.0, 0, 0, MAX_ITER)

Image(value=b'', height='480', width='640')

FloatSlider(value=1.0, continuous_update=False, description='Zoom', max=20.0, min=0.5)

IntSlider(value=0, continuous_update=False, description='Pan X', max=320, min=-320)

IntSlider(value=0, continuous_update=False, description='Pan Y', max=240, min=-240)

IntSlider(value=30, continuous_update=False, description='Max Iter', max=63, min=1)

Label(value='Press Apply View to render.')

Output()